<a href="https://colab.research.google.com/github/mathis-br/Embedding-a-Fermioninc-Hamiltonian-into-an-ISING-problem/blob/main/Estructure_to_ISING.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### In this Notebook, one provide a ressource estimation of the mapping of the fermionic hamiltonian to an Ising Problem, prvided in the following article : https://arxiv.org/pdf/1706.00271

In [15]:
# !pip install qiskit qiskit-nature pyscf qiskit-algorithms qiskit-aer matplotlib #Imports

In [16]:
### IMPORTS ###

import matplotlib.pyplot as plt
import numpy as np
from qiskit.primitives import StatevectorEstimator
from qiskit_algorithms import VQE, NumPyMinimumEigensolver
from qiskit_algorithms.optimizers import SLSQP
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import JordanWignerMapper, BravyiKitaevMapper, ParityMapper
from qiskit_nature.second_q.circuit.library import HartreeFock, UCCSD # Added UCCSD
from qiskit import transpile, QuantumCircuit
import sys
np.in1d = np.isin
from qiskit.quantum_info import SparsePauliOp
from qiskit_nature.second_q.mappers import TaperedQubitMapper
from qiskit.quantum_info import Z2Symmetries
from qiskit.visualization import plot_histogram
from IPython.display import display
from qiskit_aer import AerSimulator
from qiskit.transpiler import PassManager
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.transpiler.passes import CommutativeCancellation, Optimize1qGatesDecomposition
import itertools


In [17]:
# Paramètres :

BASIS_GATES = ['x', 'sx', 'rz', 'cz']
SEED = 42
OPT_LEVEL = 3
pm = PassManager([CommutativeCancellation(), Optimize1qGatesDecomposition()])

Fonctions Auxiliaires

In [19]:
# # ==============================================================================
# # FONCTION AUXILIAIRE : SIMULATION ET CHANGEMENT DE BASE
# # ==============================================================================


# def compare_circuits_bases(qc1, qc2, basis='Z', n_shots=8192, top_n=8, indices_hf=[0, 1, 6, 7], labels=['Modèle 1', 'Modèle 2']):
#     """
#     Simule qc1 et qc2 dans la base demandée, calcule la fidélité statistique,
#     et renvoie l'objet figure du graphique.
#     """
#     num_q = qc1.num_qubits

#     def prepare_meas_circuit(original_qc):
#         qc_m = QuantumCircuit(num_q)

#         # --- NOUVEAU : SUPERPOSITION POUR ISING ---
#         # On remplace l'état classique par une superposition totale
#         qc_m.h(range(num_q))

#         qc_m.barrier()
#         qc_m.compose(original_qc, inplace=True)
#         qc_m.barrier()

#         # --- CHANGEMENT DE BASE ---
#         # Si tu testes l'Ising, les bases X ou Y sont OBLIGATOIRES pour voir quelque chose
#         if basis == 'X':
#             qc_m.h(range(num_q))
#         elif basis == 'Y':
#             qc_m.sdg(range(num_q))
#             qc_m.h(range(num_q))

#         qc_m.measure_all()
#         return qc_m

#     qc1_m = prepare_meas_circuit(qc1)
#     qc2_m = prepare_meas_circuit(qc2)

#     # --- 2. SIMULATION ---
#     backend = AerSimulator(method='automatic')
#     circs_t = transpile([qc1_m, qc2_m], backend)
#     job = backend.run(circs_t, shots=n_shots)
#     results = job.result()

#     counts1 = results.get_counts(0)
#     counts2 = results.get_counts(1)

#     # GARDE-FOU 1 : Vérification des données
#     if not counts1 or not counts2:
#         print(f"❌ [ERREUR] Le simulateur n'a rien renvoyé pour la base {basis}.")
#         return None, 0.0

#     # --- 3. CALCUL DE LA FIDÉLITÉ (Bhattacharyya) ---
#     all_keys = set(counts1.keys()).union(set(counts2.keys()))
#     fid_val = sum(np.sqrt((counts1.get(k, 0) / n_shots) * (counts2.get(k, 0) / n_shots)) for k in all_keys)
#     fidelity = fid_val**2

#     # --- 4. FILTRAGE POUR LE GRAPHIQUE ---
#     # On trouve les états les plus fréquents en combinant les deux modèles
#     combined = {k: counts1.get(k, 0) + counts2.get(k, 0) for k in all_keys}
#     best_keys = sorted(combined, key=combined.get, reverse=True)[:top_n]

#     # GARDE-FOU 2 : Si aucun état dominant n'émerge
#     if not best_keys:
#         return None, fidelity

#     final_data1 = {k: (counts1.get(k, 0) / n_shots) * 100 for k in best_keys}
#     final_data2 = {k: (counts2.get(k, 0) / n_shots) * 100 for k in best_keys}

#     # --- 5. CRÉATION DU GRAPHIQUE ---
#     title = f"Base {basis} | Fidélité ({labels[0]} vs {labels[1]}) : {fidelity*100:.2f}%"
#     fig = plot_histogram(
#         [final_data1, final_data2],
#         legend=labels,
#         title=title,
#         figsize=(16, 7),
#         bar_labels=True,
#         color=['#1f77b4', '#ff7f0e'] # Couleurs fixes
#     )

#     return fig, fidelity


# # ==============================================================================
# # FONCTION PRINCIPALE : COMPARAISON ABSOLUE
# # ==============================================================================
# def comparaison_absolue(q1, q2, N=50000, indices_hf=[0, 1, 6, 7], labels=['qc_1', 'qc_2'], bases = ['X', 'Y', 'Z']):
#     """
#     Lance l'analyse sur les 3 bases (X, Y, Z) et gère l'affichage propre dans Jupyter.
#     """

#     for b in bases:
#         print(f"\n{'-'*65}")
#         print(f"🚀 Simulation en base {b} ({N} shots)...")

#         fig, fid = compare_circuits_bases(
#             qc1=q1,
#             qc2=q2,
#             basis=b,
#             n_shots=N,
#             top_n=8,
#             indices_hf=indices_hf,
#             labels=labels
#         )

#         if fig is not None:
#             print(f"✅ Fidélité obtenue : {fid*100:.2f}%")
#             display(fig)
#             plt.close(fig) # Ferme la figure en mémoire pour ne pas saturer le notebook
#         else:
#             print(f"⏭️ Affichage ignoré pour la base {b} (données insuffisantes).")
#         print(f"{'-'*65}")




In [20]:
def analyse(qc, name="Circuit"):
    """
    Rendu épuré des métriques d'un circuit.
    """
    depth = qc.depth()
    ops = qc.count_ops()
    sorted_ops = sorted(ops.items(), key=lambda item: item[1], reverse=True)

    # Paramètres de style
    title = f" {name.upper()} "
    width = 35

    print(f"\n{title:-^{width}}")
    print(f"DEPTH : {depth}")
    print("-" * width)
    print(f"{'GATE':<18} | {'COUNT':<10}")
    print("=" * width)  # Séparateur entre colonnes et données

    if not sorted_ops:
        print(f"{'EMPTY':<18} | {'0':<10}")
    else:
        for gate, count in sorted_ops:
            print(f"{gate:<18} | {count:<10}")

    print("-" * width + "\n")

In [21]:
# Metriques d'un Hamiltonien

def analyse_H(H):
    print(f"Nombre de qubits : {H.num_qubits}")
    print(f"Nombre de termes (PS) : {len(H)}")

### I. A frist toy example, H2

1/ Fermioninc Hamiltonian of H2, with 2 qubits

In [22]:
# 1. Define the chemical system: H2
mol_geom = """
  H 0. 0. 0.
  H 0.  0. -0.8
"""

# Using STO-3G to keep the qubit count manageable for UCCSD
driver = PySCFDriver(atom=mol_geom.strip(), basis="sto-3g", charge=0, spin=0)
H2 = driver.run()

mapperJW = JordanWignerMapper()
mapperBK = BravyiKitaevMapper()
mapperParity = ParityMapper()
def get_tapered_hamiltonian(mapper, problem = H2, num_symmetries=2, tapering_values=[1, -1]):
    """
    Réduit un Hamiltonien en utilisant le 'Tapering' (exploitation des symétries Z2).

    Args:
        mapper: Le mapper de base (ex: JordanWignerMapper).
        problem: L'objet problème (contenant l'Hamiltonien).
        num_symmetries: Nombre de symétries à appliquer pour réduire le nombre de qubits.
        tapering_values: Valeurs propres des symétries (secteurs de parité).

    Returns:
        SparsePauliOp: L'Hamiltonien réduit (tapered).
    """
    # 1. Mapper l'Hamiltonien vers l'espace qubit brut
    qubit_op = mapper.map(problem.hamiltonian.second_q_op())

    # 2. Trouver toutes les symétries Z2 disponibles
    z2_syms = Z2Symmetries.find_z2_symmetries(qubit_op)

    # 3. Sécuriser le découpage (ne pas demander plus de symétries qu'il n'en existe)
    n_to_use = min(num_symmetries, len(z2_syms.symmetries))

    # 4. Construire l'objet Z2Symmetries avec le sous-ensemble choisi
    reduced_syms = Z2Symmetries(
        symmetries=z2_syms.symmetries[:n_to_use],
        sq_paulis=z2_syms.sq_paulis[:n_to_use],
        sq_list=z2_syms.sq_list[:n_to_use],
        tapering_values=tapering_values[:n_to_use]
    )

    # 5. Créer le TaperedQubitMapper
    tapered_mapper = TaperedQubitMapper(
        mapper=mapper,
        z2symmetries=reduced_syms
    )

    # 6. Appliquer la transformation finale
    return tapered_mapper.map(problem.hamiltonian.second_q_op())

H_2_hamiltonianJW = get_tapered_hamiltonian(mapperJW, problem= H2)
print(H_2_hamiltonianJW)

SparsePauliOp(['II', 'ZI', 'IZ', 'ZZ', 'YY', 'XX'],
              coeffs=[-0.95434669+0.j,  0.16251649+0.j,  0.16251649+0.j,  0.11967584+0.j,
 -0.09231339+0.j, -0.09231339+0.j])


2/ Fermionic Hamiltonian of H2 transformed into an Ising Hamiltonian, derived from  https://arxiv.org/pdf/1706.00271

In [23]:
def ISing_hamiltonian(hamiltonian: SparsePauliOp) -> SparsePauliOp:
    """
    Transforme un Hamiltonien k-local contenant (X, Y, Z) en un Hamiltonien
    Ising diagonal (uniquement Z) dans un espace de r*n qubits. avec r = 2

    Args:
        hamiltonian (SparsePauliOp): L'opérateur qubit d'origine (n qubits).
        r (int): Le nombre de copies de l'espace (défaut = 2).

    Returns:
        SparsePauliOp: Le Hamiltonien mappé sur r*n qubits.
    """
    n = hamiltonian.num_qubits
    rn = 2 * n

    # Pour r=2, on utilise la copie j=0 et la copie k=1
    j, k = 0, 1

    # Identité globale dans le nouvel espace rn
    I_rn = SparsePauliOp(["I" * rn], [1.0])

    def mapped_single_pauli(pauli_char: str, i: int) -> SparsePauliOp:
        """Applique l'équation (3) pour un qubit i donné."""
        idx_j = j * n + i
        idx_k = k * n + i

        # Opérateurs Z sur les copies j et k
        Z_ij = SparsePauliOp.from_sparse_list([("Z", [idx_j], 1.0)], num_qubits=rn)
        Z_ik = SparsePauliOp.from_sparse_list([("Z", [idx_k], 1.0)], num_qubits=rn)

        # On assume S'(j)S'(k) = 1 (choix du sous-espace de jauge trivial)
        if pauli_char == 'X':
            return 0.5 * (I_rn - Z_ij.compose(Z_ik))
        elif pauli_char == 'Y':
            return 0.5j * (Z_ik - Z_ij)
        elif pauli_char == 'Z':
            return 0.5 * (Z_ij + Z_ik)
        elif pauli_char == 'I':
            # Attention : même l'identité doit être mappée (elle devient un projecteur)
            return 1* (I_rn + Z_ij.compose(Z_ik))
        else:
            raise ValueError(f"Opérateur inconnu : {pauli_char}")

    # Initialisation du Hamiltonien final à 0
    mapped_hamiltonian = SparsePauliOp(["I" * rn], [0.0])

    for pauli, coeff in zip(hamiltonian.paulis, hamiltonian.coeffs):
        pauli_str = pauli.to_label()

        # Terme multiplicatif neutre pour la construction
        term_op = SparsePauliOp(["I" * rn], [1.0])

        # L'ordre de Qiskit (to_label) place le qubit 0 tout à droite de la chaîne.
        # On inverse la chaîne pour que enumerate(i) corresponde bien au qubit i.
        for i, char in enumerate(reversed(pauli_str)):
            mapped_op = mapped_single_pauli(char, i)
            # On tensorise (multiplie) les opérateurs pour reconstruire la chaîne
            term_op = term_op.compose(mapped_op)

        # On applique le coefficient (les "j" des Y vont s'annuler/devenir réels ici)
        mapped_hamiltonian += coeff * term_op

    # On rassemble les termes similaires pour un résultat propre
    return mapped_hamiltonian.simplify()

In [24]:
isingH2 = ISing_hamiltonian(H_2_hamiltonianJW)

In [25]:
analyse_H(isingH2)

Nombre de qubits : 4
Nombre de termes (PS) : 16


### II. A second example, LiH

1/ Fermionin Hamiltonian of LiH : Tappered, 8 qubits

In [27]:
# 1. Define the chemical system: LiH
mol_geom = """
  Li 0. 0. 0.
  H 0.  0. 1.5949
"""

# Using STO-3G to keep the qubit count manageable for UCCSD
driver = PySCFDriver(atom=mol_geom.strip(), basis="sto-3g", charge=0, spin=0)
LiH = driver.run()


def get_tapered_hamiltonian_auto(mapper, problem):
    """
    Méthode robuste : Qiskit trouve les symétries ET le bon secteur
    (tapering_values) correspondant au nombre de particules.
    """
    # 1. Obtenir le mapper "tapered" directement depuis le problème
    # Cela calcule automatiquement les symétries et les secteurs (1, -1)
    tapered_mapper = problem.get_tapered_mapper(mapper)

    # 2. Mapper l'Hamiltonien
    return tapered_mapper.map(problem.hamiltonian.second_q_op())
LiH_JW = get_tapered_hamiltonian_auto(mapperJW, LiH)
analyse_H(LiH_JW)

Nombre de qubits : 8
Nombre de termes (PS) : 558


2/ Ferminic Hamiltonian of LiH to Ising

In [28]:
# Il y a 8 qubits, donc chaque terme de la chaine de Pauli donne 2^8 = 256 termes/PS.
# --> Nb total de terme = Nb de PS * 2^Nqubits. Pour LiH ça donne 142848 avant la simplification
# (certaines PS du nouveau Hamiltonien sont identiques)

ising_LiH = ISing_hamiltonian(LiH_JW)

analyse_H(ising_LiH)

Nombre de qubits : 16
Nombre de termes (PS) : 33920


### Conclusion : For LiH, one now need to transpie 34K paulis ZZZ.ZZZ strings (Ising) instead of 558 for the tapered standart Fermioninc Hamiltonian into the JW Transformation. It is already to much, but let's have a look at the gate count required

### Transpilation of a given Hamiltonian into a circuit : H -> $e^{iHt}$

In [29]:
def transpilation(H):
    """
    Transpilation standard Qiskit au niveau d'optimisation maximum.
    """

    num_qubits = H.num_qubits
    # 1. Préparation du Hamiltonien et du circuit
    H_real = H.simplify() # Remplacez par force_real_op(H.simplify()) si nécessaire
    qc = QuantumCircuit(num_qubits)

    # 2. Ajout de l'évolution temporelle
    evo_gate = PauliEvolutionGate(H_real, time=1.0)
    qc.append(evo_gate, range(num_qubits))

    # 3. Décomposition pour libérer les portes internes
    qc_unrolled = qc.decompose()

    # 4. Transpilation niveau 3
    qc_opt = transpile(
        qc_unrolled,
        basis_gates=BASIS_GATES,
        optimization_level=3,
        seed_transpiler=SEED
    )

    return qc_opt


def transpilation_pm(H, optlvl = 3):
    """
    Transpilation agressive avec PassManager, forçant le retour strict aux BASIS_GATES.
    """
    num_qubits = H.num_qubits
    H_real = H.simplify()
    qc_base = QuantumCircuit(num_qubits)

    qc_base.compose(PauliEvolutionGate(H_real, time=1.0).definition, inplace=True)

    # 1. Transpilation initiale
    qc_transpiled = transpile(
        qc_base,
        basis_gates=BASIS_GATES,
        optimization_level=optlvl,
        seed_transpiler=SEED
    )

    # 2. Passage au rouleau compresseur (qui va générer des u2/u3)
    qc_pm_opt = pm.run(qc_transpiled)

    # 3. FIX : Re-transpilation finale pour forcer la synthèse vers tes BASIS_GATES
    qc_final = transpile(
        qc_pm_opt,
        basis_gates=BASIS_GATES,
        optimization_level=3,
        seed_transpiler=SEED
    )

    return qc_final

In [30]:
qc_LiH_ising = transpilation(ising_LiH)   # Dure 3min

In [31]:
qc_LiH_JW = transpilation_pm(LiH_JW)  # 0.2s

In [32]:
analyse(qc_LiH_ising, 'ISING Opt level = 3')
# analyse(qc_LiH_ising_pm, 'ISING puis PM')
analyse(qc_LiH_JW, 'JW')


------- ISING OPT LEVEL = 3 -------
DEPTH : 1554872
-----------------------------------
GATE               | COUNT     
rz                 | 732892    
sx                 | 467173    
cz                 | 436364    
x                  | 343272    
-----------------------------------


--------------- JW ----------------
DEPTH : 9400
-----------------------------------
GATE               | COUNT     
rz                 | 4461      
sx                 | 4410      
cz                 | 3012      
x                  | 1360      
-----------------------------------



### Conclusion : Le Mur de la Réalité Matérielle et Combinatoire

Bien que cette idée paraisse extrêmement prometteuse sur le papier, elle se heurte de plein fouet à la dure réalité de la complexité algorithmique et matérielle :

* **L'explosion combinatoire :** Le nombre de termes à évaluer croît de manière exponentielle au fur et à mesure que la taille du système augmente.
* **Le goulot d'étranglement de la topologie :** Cette approche exige une connectivité matérielle totale (*all-to-all*). Sur les architectures NISQ actuelles (souvent planaires avec une connectivité locale), cela impose un routage catastrophique qui fait exploser la profondeur du circuit avec l'ajout massif de portes SWAP, noyant le signal dans la décohérence.

**Le verdict est sans appel :** Tenter d'exprimer des corrélations purement quantiques par le prisme de contraintes classiques — comme c'est le cas avec la formulation d'un Hamiltonien fermionique sous forme de modèle d'Ising — est une impasse pour espérer atteindre l'avantage quantique, ni même une utilité quantique.  

Pour simuler efficacement la chimie quantique, un algorithme doit exploiter nativement l'intrication et les symétries physiques du système, plutôt que d'essayer de forcer la nature quantique dans un carcan combinatoire classique.